# Image forgery detection: training and evaluation

Run on Google Colab with a GPU (Runtime → Change runtime type → T4 GPU).

Six experiments, each evaluated on a held-out test set containing **both authentic and forged images**:

| Experiment | Model | Input | Loss |
|---|---|---|---|
| `unet_bce` | U-Net | RGB | BCE |
| `unet_combined` | U-Net | RGB | BCE + Dice + IoU |
| `unet_deepsup_combined` | U-Net, 5 output heads | RGB | BCE + Dice + IoU on every head |
| `unet_combined_ela` | U-Net | RGB + ELA | BCE + Dice + IoU |
| `swin_unet_combined` | Swin-UNet (pretrained Swin-T encoder) | RGB | BCE + Dice + IoU |
| `swin_unet_combined_ela` | Swin-UNet | RGB + ELA | BCE + Dice + IoU |

Metrics: **Dice / IoU** (how well forged regions are outlined, forged images only) and **accuracy / F1 / ROC-AUC** (whether each whole image is authentic or forged).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/GunalKarthikeyanS/Image-Forgery-Detection.git
%cd Image-Forgery-Detection
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "src")

In [ ]:
# Change these if your data lives elsewhere on Drive
DATA = "/content/drive/MyDrive/image_forgery/data/recodai"
RUNS = "/content/drive/MyDrive/image_forgery/runs"   # checkpoints and results are saved here

SIZE = 448     # 448 works for both U-Net and Swin (Swin needs 224 or 448)
BATCH = 4
EPOCHS = 15

## 2. Data splits

Forged images are matched to their masks by file name; every split contains authentic and forged images.

In [ ]:
import torch
from torch.utils.data import DataLoader
from dataset import make_splits, balance, ForgeryDataset

splits = make_splits(f"{DATA}/train_images/authentic",
                     f"{DATA}/train_images/forged",
                     f"{DATA}/train_masks")
train_items = balance(splits["train"])
device = "cuda"

_loaders = {}
def loaders(use_ela):
    if use_ela not in _loaders:
        _loaders[use_ela] = (
            DataLoader(ForgeryDataset(train_items, SIZE, augment=True, use_ela=use_ela),
                       BATCH, shuffle=True, drop_last=True, num_workers=2),
            DataLoader(ForgeryDataset(splits["val"], SIZE, use_ela=use_ela), BATCH, num_workers=2),
            DataLoader(ForgeryDataset(splits["test"], SIZE, use_ela=use_ela), BATCH, num_workers=2))
    return _loaders[use_ela]

## 3. Check images, ELA and masks

Each mask should outline a region of the image above it. Forged regions often look different in the ELA row.

In [ ]:
import matplotlib.pyplot as plt

examples = [it for it in splits["train"] if it[1] is not None][:4]
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for j, item in enumerate(examples):
    x, y, _ = ForgeryDataset([item], SIZE, use_ela=True)[0]
    axes[0, j].imshow(x[:3].permute(1, 2, 0)); axes[0, j].set_title(item[0].name)
    axes[1, j].imshow(x[3:].permute(1, 2, 0)); axes[1, j].set_title("ELA")
    axes[2, j].imshow(y[0], cmap="gray"); axes[2, j].set_title("mask")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Experiments

Each experiment trains for `EPOCHS` epochs, keeps the checkpoint with the best validation Dice, then scores it on the test set. Results are saved to Drive, so finished experiments are skipped when you rerun the notebook after a disconnect. An experiment interrupted midway restarts from the beginning.

Run them in order; the first three take priority if time is short.

In [ ]:
import json, os
from models import UNet, UNetDeepSup, SwinUNet
from losses import CombinedLoss, DeepSupervisionLoss
from train import train
from metrics import evaluate

EXPERIMENTS = {   # name: (model, loss, use ELA)
    "unet_bce":               (lambda: UNet(),                              lambda: CombinedLoss(1, 0, 0), False),
    "unet_combined":          (lambda: UNet(),                              lambda: CombinedLoss(),        False),
    "unet_deepsup_combined":  (lambda: UNetDeepSup(),                       lambda: DeepSupervisionLoss(CombinedLoss()), False),
    "unet_combined_ela":      (lambda: UNet(in_channels=6),                 lambda: CombinedLoss(),        True),
    "swin_unet_combined":     (lambda: SwinUNet(img_size=SIZE),             lambda: CombinedLoss(),        False),
    "swin_unet_combined_ela": (lambda: SwinUNet(in_channels=6, img_size=SIZE), lambda: CombinedLoss(),     True),
}

def run(name):
    out = f"{RUNS}/{name}"
    if os.path.exists(f"{out}/test.json"):
        print(f"{name}: already done, skipping")
        return
    torch.manual_seed(42)
    make_model, make_loss, use_ela = EXPERIMENTS[name]
    train_dl, val_dl, test_dl = loaders(use_ela)
    model = make_model().to(device)
    print(f"{name}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    train(model, train_dl, val_dl, make_loss(), device, out, epochs=EPOCHS)
    result = evaluate(model, test_dl, device)
    json.dump(result, open(f"{out}/test.json", "w"), indent=1)
    print(name, result)

In [ ]:
run("unet_bce")

In [ ]:
run("unet_combined")

In [ ]:
run("unet_deepsup_combined")

In [ ]:
run("unet_combined_ela")

In [ ]:
run("swin_unet_combined")

In [ ]:
run("swin_unet_combined_ela")

## 5. Test results

In [ ]:
import pandas as pd

rows = []
for name in EXPERIMENTS:
    path = f"{RUNS}/{name}/test.json"
    if os.path.exists(path):
        rows.append({"experiment": name, **json.load(open(path))})
results = pd.DataFrame(rows)
results.round(3)

## 6. Example predictions (best model)

In [ ]:
best = max(rows, key=lambda r: r["dice"])["experiment"]
make_model, _, use_ela = EXPERIMENTS[best]
model = make_model().to(device)
model.load_state_dict(torch.load(f"{RUNS}/{best}/best.pth", map_location=device))
model.eval()

items = ([it for it in splits["test"] if it[1] is not None][:3] +
         [it for it in splits["test"] if it[1] is None][:2])
fig, axes = plt.subplots(len(items), 3, figsize=(10, 3.3 * len(items)))
for r, item in enumerate(items):
    x, y, forged = ForgeryDataset([item], SIZE, use_ela=use_ela)[0]
    with torch.no_grad():
        out = model(x[None].to(device))
        out = out[-1] if isinstance(out, list) else out
        pred = torch.sigmoid(out)[0, 0].cpu()
    panels = [(x[:3].permute(1, 2, 0), "forged" if forged else "authentic"),
              (y[0], "true mask"), (pred, "predicted")]
    for c, (img, title) in enumerate(panels):
        axes[r, c].imshow(img, cmap=None if c == 0 else "gray", vmin=0, vmax=1)
        axes[r, c].set_title(title); axes[r, c].axis("off")
plt.suptitle(f"Best model: {best}")
plt.tight_layout()
plt.savefig(f"{RUNS}/predictions.png", dpi=120)
plt.show()